In [8]:
import logging
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd

dev = pd.read_excel(Path(r"C:\Users\gkilber\Downloads\moxy_new_securities_compare.xlsx"), sheet_name="dev")
prd = pd.read_excel(Path(r"C:\Users\gkilber\Downloads\moxy_new_securities_compare.xlsx"), sheet_name="prd")

In [9]:
sql = """
select distinct symbol
from prd
where symbol not in (select distinct ticker from dev)
"""

duckdb.sql(sql)

┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                     symbol                                                      │
│                                                     varchar                                                     │
├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 04899D065                                                                                                       │
│ 302319999999999981680545621476110386977302679856806959656717536297960792016556861816928402834748783793966940160 │
│ 604CVR019                                                                                                       │
│ CWTR                                                                                                            │
│ GLPEY                                                                 

In [38]:
# UPDATE YAML, ADD POLICIES

import sys
from pathlib import Path

import yaml

files = Path.cwd().glob("models/base/**/*yml")
file_list = list(files)

# Load column list
with open("tmp.txt") as file:
    col_list = [line.rstrip() for line in file]

for idx, filepath in enumerate(file_list, start=1):
    is_changed = False
    print(f"{idx}/{len(file_list)} {Path(filepath).as_posix()}")
    with open(filepath, encoding="utf8") as file:
        yml = yaml.safe_load(file)

        if yml.get("models"):
            for tbl in yml.get("models"):
                for col in tbl.get("columns", []):
                    if col.get("name", "") in col_list:

                        # if col.get("meta", {}).get("masking_policy", "") == "" and col.get("data_type", "") != "BOOLEAN":
                        #     # Add the policy
                        #     print(f"Need to add policy to col {col.get('name')}")
                        #     col.setdefault("meta", {})["masking_policy"] = "mp_client_pii"

                        #     is_changed = True

                        if "mp_client_pii" in col.get("meta", {}).get("masking_policy", ""):
                            print(f"Need to update policy to col {col.get('name')}")
                            if col.get("data_type", "") == "DATE":
                                col.setdefault("meta", {})["masking_policy"] = "mp_client_pii_date"
                            elif col.get("data_type", "") == "VARCHAR":
                                col.setdefault("meta", {})["masking_policy"] = "mp_client_pii_text"
                            elif col.get("data_type", "") == "FLOAT":
                                col.setdefault("meta", {})["masking_policy"] = "mp_client_pii_float"
                            elif col.get("data_type", "") == "NUMBER":
                                col.setdefault("meta", {})["masking_policy"] = "mp_client_pii_number"
                            elif col.get("data_type", "") == "BOOLEAN":
                                col.pop("meta")
                            else:
                                col.setdefault("meta", {})["masking_policy"] = "mp_client_pii_text"
                            is_changed = True

                        if "mp_associate_pii" in col.get("meta", {}).get("masking_policy", ""):
                            print(f"Need to update policy to col {col.get('name')}")
                            if col.get("data_type", "") == "DATE":
                                col.setdefault("meta", {})["masking_policy"] = "mp_associate_pii_date"
                            elif col.get("data_type", "") == "VARCHAR":
                                col.setdefault("meta", {})["masking_policy"] = "mp_associate_pii_text"
                            elif col.get("data_type", "") == "FLOAT":
                                col.setdefault("meta", {})["masking_policy"] = "mp_associate_pii_float"
                            elif col.get("data_type", "") == "NUMBER":
                                col.setdefault("meta", {})["masking_policy"] = "mp_associate_pii_number"
                            elif col.get("data_type", "") == "BOOLEAN":
                                col.pop("meta")
                            else:
                                col.setdefault("meta", {})["masking_policy"] = "mp_associate_pii_text"
                            is_changed = True

    if is_changed:
        with open(filepath, "w") as file:
            yaml.dump(yml, file, sort_keys=False)
            print("Saved yaml")

    # if idx >= 8:
    #     break


1/235 c:/repos/dbt-snf/models/base/_reporting__sources.yml
2/235 c:/repos/dbt-snf/models/base/activebatch/_activebatch__models.yml
3/235 c:/repos/dbt-snf/models/base/activebatch/_activebatch__sources.yml
4/235 c:/repos/dbt-snf/models/base/active_directory/_active_directory__models.yml
Need to update policy to col DATEOFBIRTH
Need to update policy to col DATE_OF_BIRTH
Need to update policy to col DATEOFBIRTH
Need to update policy to col DATEOFBIRTH
Need to update policy to col DATE_OF_BIRTH
Saved yaml
5/235 c:/repos/dbt-snf/models/base/active_directory/_active_directory__sources.yml
6/235 c:/repos/dbt-snf/models/base/adp/_adp__models.yml
Need to update policy to col BIRTH_DATE
Need to update policy to col BIRTH_DATE
Need to update policy to col BIRTH_DATE
Saved yaml
7/235 c:/repos/dbt-snf/models/base/adp/_adp__sources.yml
8/235 c:/repos/dbt-snf/models/base/alteryx_gallery/_alteryx_gallery__models.yml
9/235 c:/repos/dbt-snf/models/base/alteryx_gallery/_alteryx_gallery__sources.yml
10/235